In [ ]:
# libraries
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from tensorflow import math as TFmath
import math,os,shutil
from prettytable import PrettyTable
import scipy.stats as stats
from sklearn.model_selection import train_test_split
import swanlab
from swanlab.integration.keras import SwanLabLogger

print("TensorFlow version:", tf.__version__)


gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))
print("Physical GPUs:", gpus)

if len(gpus) == 0:
    print("\n[警告] 未检测到 GPU。可能有以下原因：")
    print("1. 未安装 NVIDIA 显卡驱动或驱动版本过低。")
    print("2. 未安装 CUDA Toolkit (需 11.2) 或 cuDNN (需 8.1) - 针对 TF 2.10 Windows版。")
    print("3. 如果安装了 TF > 2.10，Windows Native GPU 支持已被移除，需使用 WSL2。")
else:

    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU 显存设置为按需分配 (Memory Growth Enable)")
    except RuntimeError as e:
        print(e)

tf.random.set_seed(256)

In [ ]:
# Full paths to training data
TRAIN_FEATURE_PATH = "../Features/Train/train_features.csv"
TRAIN_LABEL_PATH   = "../Features/Train/train_labels.csv"

# Full path to test data
TEST_FEATURE_PATH = "../Features/Test/test_features.csv"

# Output folders
figure_prepath = "../figures/"
model_prepath = "../model/siluse_block_fe/"

# Create output folders if not existed
os.makedirs(figure_prepath, exist_ok=True)
os.makedirs(model_prepath, exist_ok=True)

# Plotting configuration
figureFrontSize = 12
figureName_post = "_test.png"


In [ ]:
# 检查当前工作目录和数据文件
import os

print("当前工作目录:", os.getcwd())
print("\n检查数据文件是否存在:")
print(f"训练特征文件: {os.path.exists(TRAIN_FEATURE_PATH)} - {TRAIN_FEATURE_PATH}")
print(f"训练标签文件: {os.path.exists(TRAIN_LABEL_PATH)} - {TRAIN_LABEL_PATH}")
print(f"测试特征文件: {os.path.exists(TEST_FEATURE_PATH)} - {TEST_FEATURE_PATH}")

# 列出上级目录的内容
parent_dir = os.path.abspath("..")
print(f"\n上级目录 ({parent_dir}) 的内容:")
if os.path.exists(parent_dir):
    print(os.listdir(parent_dir))
    
# 检查 Features 文件夹
features_dir = os.path.join(parent_dir, "Features")
if os.path.exists(features_dir):
    print(f"\nFeatures 目录存在，内容:")
    print(os.listdir(features_dir))
else:
    print(f"\nFeatures 目录不存在: {features_dir}")

In [ ]:
### helper labels - fixed
Numchannels = 95
num_inputFeatures = Numchannels * 2 + 4 # target_gain, target_gain_tilt, EDFA_input_power_total, EDFA_output_power_total
labels = {"gainValue":'target_gain',
          "EDFA_input":'EDFA_input_power_total',
          "EDFA_output":'EDFA_output_power_total',
          "inSpectra":'EDFA_input_spectra_',
          "WSS":'DUT_WSS_activated_channel_index_',
          "result":'calculated_gain_spectra_'}
inSpectra_labels = [labels['inSpectra']+str(i).zfill(2) for i in range(0,Numchannels)]
onehot_labels = [labels['WSS']+str(i).zfill(2) for i in range(0,Numchannels)]
result_labels = [labels['result']+str(i).zfill(2) for i in range(0,Numchannels)]
preProcess_labels = [labels['EDFA_input'],labels['EDFA_output']]
preProcess_labels.extend(inSpectra_labels)

In [ ]:
def dB_to_linear(data):
  return np.power(10,data/10)

def linear_TO_Db(data):
  result = 10*np.log10(data).to_numpy()
  return result[result != -np.inf]

def linear_TO_Db_full(data):
  result = 10*np.log10(data).to_numpy()
  result[result == -np.inf] = 0
  return result

def divideZero(numerator,denominator):
  with np.errstate(divide='ignore'):
    result = numerator / denominator
    result[denominator == 0] = 0
  return result

In [ ]:

def add_shb_features(df):
    df_eng = df.copy()

    spectra_cols = [c for c in df.columns if 'EDFA_input_spectra' in c]
    wss_cols = [c for c in df.columns if 'DUT_WSS_activated_channel_index' in c]
    spectra_cols.sort()
    wss_cols.sort()

    df_eng['shb_std']  = df_eng[spectra_cols].std(axis=1)
    df_eng['shb_mean'] = df_eng[spectra_cols].mean(axis=1)
    df_eng['shb_sum']  = df_eng[spectra_cols].sum(axis=1)
    df_eng['shb_max']  = df_eng[spectra_cols].max(axis=1)

    df_eng['feat_active_channel_count'] = df_eng[wss_cols].sum(axis=1)

    indices = np.arange(len(spectra_cols))
    weighted_sum = df_eng[spectra_cols].dot(indices)
    total_p = df_eng[spectra_cols].sum(axis=1) + 1e-9
    df_eng['shb_center'] = weighted_sum / total_p

    mw_df = np.power(10, df_eng[spectra_cols] / 10.0)
    df_eng['total_power_mW'] = mw_df.sum(axis=1)

    df_eng['power_span'] = df_eng[spectra_cols].max(axis=1) - df_eng[spectra_cols].min(axis=1)
    
    return df_eng

In [ ]:
### train and loss function
def custom_loss(y_actual,y_pred):
  # calculate the loaded channel numbers for each batch
  # batch default is [batch size=32, outputchannel number]
  loaded_size = tf.dtypes.cast(TFmath.count_nonzero(y_actual), tf.float32)
  # turn unloaded y_pred prediction to zero
  y_pred_cast_unloaded_to_zero = TFmath.divide_no_nan(TFmath.multiply(y_pred,y_actual),y_actual)
  # error [unloaded,unloaded,loaded,loaded]: y_pred = [13->0,15->0,18.5,18.3], y_actual = [0,0,18.2,18.2]
  error = TFmath.abs(TFmath.subtract(y_pred_cast_unloaded_to_zero,y_actual))
  # custom_loss = (0.3+0.2) / 2
  custom_loss = TFmath.divide(TFmath.reduce_sum(error),loaded_size)
  return custom_loss

def custom_loss_L2(y_actual,y_pred):
  loaded_size = tf.dtypes.cast(TFmath.count_nonzero(y_actual), tf.float32)
  y_pred_cast_unloaded_to_zero = TFmath.divide_no_nan(TFmath.multiply(y_pred,y_actual),y_actual)
  error = TFmath.square(TFmath.subtract(y_pred_cast_unloaded_to_zero,y_actual))
  custom_loss = TFmath.sqrt(TFmath.divide(TFmath.reduce_sum(error),loaded_size))
  return custom_loss
    

In [ ]:
def se_block(input_tensor, ratio=16):

    # Input shape: (Batch, input_dim)
    input_dim = input_tensor.shape[-1]

    x = layers.Dense(input_dim // ratio, use_bias=False, activation='relu')(input_tensor)

    x = layers.Dense(input_dim, use_bias=False, activation='sigmoid')(x)
    

    return layers.Multiply()([input_tensor, x])

def designed_DNN_model(outputNum, X_data=None):


  inputs = layers.Input(shape=(num_inputFeatures,))

  target_gain = layers.Lambda(lambda x: tf.expand_dims(x[:, 0], -1), name='extract_target_gain')(inputs)
  
  x = inputs

  if X_data is not None:
      normalizer = layers.Normalization(axis=-1)
      normalizer.adapt(X_data)
      x = normalizer(inputs)

  x = se_block(x, ratio=8) 


  x = layers.Dense(num_inputFeatures, activation='silu')(x)

  x = layers.Dense(256)(x)
#   x = layers.BatchNormalization()(x)
  x = layers.Activation('silu')(x)
  x = layers.Dropout(0.1)(x) # 增加 Dropout, 比率设为 0.2
  
  # === SE-Block 2: Intermediate Feature Refinement ===
  x = se_block(x, ratio=16)

  x = layers.Dense(128)(x)
#   x = layers.BatchNormalization()(x)
  x = layers.Activation('silu')(x)
  x = layers.Dropout(0.1)(x) # 增加 Dropout

  x = layers.Dense(128)(x)
#   x = layers.BatchNormalization()(x)
  x = layers.Activation('silu')(x)
  x = layers.Dropout(0.1)(x) # 增加 Dropout

  x = layers.Dense(128)(x)
#   x = layers.BatchNormalization()(x)
  x = layers.Activation('silu')(x)
  x = layers.Dropout(0.05)(x) # 最后一层 Dropout 稍微小点

  
  residual = layers.Dense(outputNum, name='predicted_residual')(x)

  outputs = layers.Add(name='add_residual_to_target')([residual, target_gain])
  
  model = keras.Model(inputs=inputs, outputs=outputs)

  model.compile(loss=custom_loss_L2, # custom_loss
                optimizer=tf.keras.optimizers.Adam(0.001))
  return model

### debug function after train -> go to csv

In [ ]:
from sklearn.model_selection import KFold
import gc

X_train_raw = pd.read_csv(TRAIN_FEATURE_PATH).iloc[:, 3:]
y_train = pd.read_csv(TRAIN_LABEL_PATH)

print(f"Original features: {X_train_raw.shape[1]}")
print("Applying SHB features to training data...")
X_train = add_shb_features(X_train_raw)

global num_inputFeatures 
num_inputFeatures = X_train.shape[1]
print(f"Features updated. New input shape: {num_inputFeatures}")

y_train.fillna(0, inplace=True)

# K-Fold configuration
n_splits = 7
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Track the best model across all folds
best_global_val_loss = float('inf')
best_global_model_path = ""

# K-Fold Training Loop
fold_no = 1
for train_index, val_index in kf.split(X_train):

    print(f"Start Training Fold {fold_no}...")

    tf.keras.backend.clear_session()
    gc.collect()

    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    base_model = designed_DNN_model(Numchannels, X_tr)

    epochs = 300
    batch_size = 64
    if len(X_tr) > 0:
        steps_per_epoch = len(X_tr) // batch_size
    else:
        steps_per_epoch = 1 
        
    decay_steps = steps_per_epoch * epochs
    
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=decay_steps,
        alpha=0.001 
    )

    base_model.compile(
        loss=custom_loss_L2, 
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule)
    )


    fold_model_path = f"{model_prepath}/ML_example_model_fold{fold_no}.h5"
    
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=fold_model_path,
        monitor='val_loss',
        save_best_only=True, 
        mode='min',
        save_weights_only=False,
        verbose=0 
    )

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=50,
        restore_best_weights=True,
        verbose=1
    )

    print(f"   [DNN] Training with SE-Block + FE (CosineAnnealing)...")
    history_dnn = base_model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        verbose=2, 
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[checkpoint, early_stopping] # Add SwanLabLogger
    )
    print(f"   [DNN] Fold {fold_no} finished. Best model saved to {fold_model_path}")
    
    fold_no += 1

print("K-Fold Cross Validation Completed. All models are saved.")

In [ ]:

def plot_loss(indx,history,ingnoreIndex):
    plt.figure(indx)
    plt.plot(history.history['loss'][ingnoreIndex:], label='loss')
    plt.plot(history.history['val_loss'][ingnoreIndex:], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Error [gain]')
    plt.legend()
    plt.grid(True)
plot_loss(1, history_dnn, 15)

In [ ]:
# === WRAPPED PREDICTION FUNCTION ===
def run_ensemble_prediction():
    print(">>> FUNCTION START: run_ensemble_prediction")
    
    # 1. Load Test Data
    if not os.path.exists(TEST_FEATURE_PATH):
        print(f"Error: Test feature path not found: {TEST_FEATURE_PATH}")
        return

    X_test_full = pd.read_csv(TEST_FEATURE_PATH)
    X_test_raw = X_test_full.iloc[:, 5:]

    print("Applying SHB features to test data...")
    X_test = add_shb_features(X_test_raw)
    print(f"Test features shape: {X_test.shape}")

    # 2. Configuration
    n_splits = 7
    y_pred_ensemble = None
    
    #Explicitly set the model path
    ensemble_model_path = "../model/siluse_block_fe/"

    # 3. Prediction Loop
    print(f"--- Starting loop for {n_splits} folds from {ensemble_model_path} ---")
    for i in range(1, n_splits + 1):
        print(f"   Processing Fold {i}/{n_splits} ...")
        
        # --- A. DNN Prediction ---
        pred_dnn = None
        # 使用 explicitly 定义的 path
        fold_model_path = f"{ensemble_model_path}/ML_example_model_fold{i}.h5"
        
        if os.path.exists(fold_model_path):
            try:
                model_fold = tf.keras.models.load_model(fold_model_path, compile=False)
                pred_dnn = model_fold.predict(X_test, verbose=0)
            except Exception as e:
                print(f"     [!] DNN Exception fold {i}: {e}")
        else:
            print(f"     [!] DNN Model file missing: {fold_model_path}")

        # --- B. Fusion for this fold (Just DNN now) ---
        if pred_dnn is not None:
             pred_fold = pred_dnn
             print(f"     [+] Fold {i} Prediction done")
        else:
             print("     [!] No prediction for this fold")
             continue

        # Accumulate
        if y_pred_ensemble is None:
            y_pred_ensemble = pred_fold
        else:
            y_pred_ensemble += pred_fold

    # 4. Finalize
    if y_pred_ensemble is None:
        print("Error: No predictions were made.")
        return

    print("--- Loop finished. averaging and saving... ---")
    y_pred_array = y_pred_ensemble / n_splits
    
    # 5. Format Submission
    y_pred = pd.DataFrame(y_pred_array, columns=y_train.columns)
    
    wss_cols = [col for col in X_test.columns if 'dut_wss_activated_channel_index' in col.lower()]
    label_cols = [col for col in y_train.columns if 'calculated_gain_spectra' in col.lower()]
    
    # Apply Mask
    mask = X_test[wss_cols].values == 1
    y_pred = pd.DataFrame(np.where(mask, y_pred.values, np.nan), columns=label_cols)
    y_pred.fillna(0, inplace=True)
    
    # Add ID
    kaggle_ID = X_test_full.columns[0]
    y_pred.insert(0, kaggle_ID, X_test_full[kaggle_ID].values)
    
    # Save
    output_path = "../Features/Test/submission_se_block_fe.csv"
    y_pred.to_csv(output_path, index=False)
    print(f"SUCCESS: Submission saved to {output_path}")
    print(">>> FUNCTION END")

# Call the function exactly once
run_ensemble_prediction()